In [9]:
## Matrix series indicator

import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
import numpy as np
import vectorbt as vbt
import pandas as pd
import numba as nb

In [90]:
## Load datasource from deltalake and save it to local h5 file
REMOTE_HOST = "http://192.168.1.30:9000"

from deltalake import DeltaTable
def load_stock_data() -> pd.DataFrame:
    """Load stock data from delta lake, ensuring index uniqueness to avoid unstack ValueError"""
    try:
        from deltalake import DeltaTable
        storage_options = {
            "AWS_ACCESS_KEY_ID": "CzOwnLkEDXQy951AOqes",
            "AWS_SECRET_ACCESS_KEY": "fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S",
            "AWS_ENDPOINT_URL": REMOTE_HOST,
            "AWS_ALLOW_HTTP": "true",
            "AWS_EC2_METADATA_DISABLED": "true",
            "AWS_REGION": 'us-east-1',
            "aws_conditional_put": "etag",
        }        
        dt = DeltaTable("s3://delta-table-storage/stocks", storage_options=storage_options)
        watchlist_symbols = ["VHM", "GVR", "NKG", "PAN", "ANV"]

        now = pd.Timestamp.now()
        start_date = now - pd.DateOffset(years=2)
        # Load all, then filter
        df = dt.to_pandas(
            filters=[("date", ">=", start_date), ("symbol", "in", watchlist_symbols)], 
            columns=["symbol", "date", "close", "open", "high", "low", "volume"]
        )
        
        # Remove duplicate (date, symbol) rows by keeping the last occurrence
        df = df.drop_duplicates(subset=["date", "symbol"], keep="last")
        # Convert to the same format as H5 store
        df = df.set_index(["date", "symbol"])
        stocks = df.unstack(level=1).bfill().ffill()
        
        print("Successfully loaded data from delta lake")
        return stocks
        
    except Exception as e:
        print(f"Error loading from delta lake: {e}")
        raise

df = load_stock_data()
df.tail(5)

Successfully loaded data from delta lake


close                                        open             \
symbol            ANV    GVR    NKG        PAN         VHM    ANV        GVR   
date                                                                           
2025-12-23  27.299999  26.25  15.15  28.100000  114.900002  28.00  26.650000   
2025-12-24  26.850000  26.00  15.20  28.150000  122.800003  27.35  26.400000   
2025-12-25  26.700001  25.65  14.95  28.299999  114.300003  26.85  26.000000   
2025-12-26  27.000000  25.50  15.30  27.799999  110.000000  26.60  25.549999   
2025-12-29  26.450001  26.00  15.20  27.500000  117.699997  27.00  25.600000   

                                      ...        low                    \
symbol        NKG    PAN         VHM  ...        ANV        GVR    NKG   
date                                  ...                                
2025-12-23  15.35  28.10  108.500000  ...  27.299999  26.250000  15.10   
2025-12-24  15.15  27.90  117.300003  ...  26.850000  25.900000  15.10   
2025-12-25  15.20  28.15  124.000000  ...  26.700001  25.650000  14.95   
2025-12-26  15.30  28.00  106.300003  ...  26.350000  25.200001  15.00   
2025-12-29  15.40  27.75  114.500000  ...  26.400000  25.600000  15.15   

                                      volume                                   \
symbol            PAN         VHM        ANV        GVR        NKG        PAN   
date                                                                            
2025-12-23  27.799999  107.599998   856500.0  1072000.0  3778900.0   518700.0   
2025-12-24  27.850000  114.500000  1115100.0  1436000.0  1994900.0   480800.0   
2025-12-25  27.900000  114.300003  1027100.0  1031900.0  3086800.0  1040100.0   
2025-12-26  27.549999  106.300003   918600.0  1406500.0  6410800.0   850800.0   
2025-12-29  27.299999  114.500000   708700.0  1106300.0  1935600.0   521000.0   

                        
symbol             VHM  
date                    
2025-12-23   7781700.0  
2025-12-24   9949600.0  
2025-12-25  10651700.0  
2025-12-26  16551200.0  
2025-12-29   7324300.0  

[5 rows x 25 columns]

In [116]:
pd.set_option('display.max_rows', 200)

def wilders(s: pd.Series, n: int) -> pd.Series:
    """AFL Wilders smoothing ~ EMA with alpha=1/n, adjust=False."""
    return s.ewm(alpha=1 / n, adjust=False).mean()

def wilders_ndarray(s: np.ndarray, n: int) -> np.ndarray:
    rows, cols = s.shape
    result = np.zeros_like(s)
    for col in range(cols):
        # pandas ewm can't be used on numpy array, so use pandas.Series per column:
        result[:, col] = pd.Series(s[:, col]).ewm(alpha=1 / n, adjust=False).mean().values
    return result

def matrix_series(close: np.ndarray, high: np.ndarray, low: np.ndarray, price_period: int = 20, supResPeriod: int = 50, supResPercentage: int = 100, smoother: int = 5):
  
    Osc = vbt.IndicatorFactory.from_talib('CCI').run(high, low, close, timeperiod=price_period).real

    Value1 = Osc
    Value2 = vbt.IndicatorFactory.from_talib('MAX').run(Value1, timeperiod=supResPeriod).real
    Value3 = vbt.IndicatorFactory.from_talib('MIN').run(Value1, timeperiod=supResPeriod).real
    Value4 = Value2 - Value3
    Value5 = Value4 * (supResPercentage / 100.0)

    ResistanceLine = Value3 + Value5
    SupportLine = Value2 - Value5

    # Signal line 
    ys1 = (high + low + close * 2) / 4.0
    # rk3 = vbt.MA.run(ys1, window=smoother, ewm=True).ma
    rk3 = vbt.IndicatorFactory.from_talib('EMA').run(ys1, timeperiod=smoother).real.to_numpy()
    rk4 = vbt.IndicatorFactory.from_talib('STDDEV').run(ys1, timeperiod=smoother, nbdev=1).real.to_numpy()
    rk5 = (ys1 - rk3) * 200.0 / rk4

    # rk6 = vbt.MA.run(rk5, window=smoother, ewm=True).ma
    rk6 = vbt.IndicatorFactory.from_talib('EMA').run(rk5, timeperiod=smoother).real
    UP_line = vbt.IndicatorFactory.from_talib('EMA').run(rk6, timeperiod=smoother).real
    DOWN_line = vbt.IndicatorFactory.from_talib('EMA').run(UP_line, timeperiod=smoother).real

    # Candle OHLC
    Hh = UP_line.where(UP_line < DOWN_line, DOWN_line)      # High = min(up, down)
    Ll = DOWN_line.where(UP_line < DOWN_line, UP_line)      # Low = max(up, down)  
    
    return Hh, Ll, SupportLine, ResistanceLine

## Matrix series indicator
matrix_series_indicator = vbt.IndicatorFactory(
    class_name='MatrixSeries',
    short_name='matrix_series',
    input_names=['close', 'high', 'low'],
    param_names=['price_period', 'supResPeriod', 'supResPercentage', 'smoother'],
    output_names=['hh', 'll', 'support_line', 'resistance_line']
).from_apply_func(matrix_series)

matrix_series = matrix_series_indicator.run(df.close.round(2), df.high.round(2), df.low.round(2), 
    price_period=20, supResPeriod=50, supResPercentage=100, smoother=5)

up = matrix_series.hh
down = matrix_series.ll
support_line = matrix_series.support_line
resistance_line = matrix_series.resistance_line

resistance_line.tail(100)

matrix_series_price_period              20                          \
matrix_series_supResPeriod              50                           
matrix_series_supResPercentage         100                           
matrix_series_smoother                   5                           
symbol                                 ANV         GVR         NKG   
date                                                                 
2025-08-08                      283.251541  191.580069  251.181634   
2025-08-11                      283.251541  191.580069  251.181634   
2025-08-12                      283.251541  191.580069  251.181634   
2025-08-13                      283.251541  191.580069  251.181634   
2025-08-14                      283.251541  191.580069  251.181634   
2025-08-15                      283.251541  191.580069  251.181634   
2025-08-18                      283.251541  191.580069  251.181634   
2025-08-19                      283.251541  191.580069  251.181634   
2025-08-20                      283.251541  191.580069  251.181634   
2025-08-21                      283.251541  191.580069  251.181634   
2025-08-22                      283.251541  191.580069  251.181634   
2025-08-25                      283.251541  191.580069  251.181634   
2025-08-26                      283.251541  191.580069  251.181634   
2025-08-27                      283.251541  191.580069  251.181634   
2025-08-28                      283.251541  191.580069  251.181634   
2025-08-29                      283.251541  191.580069  251.181634   
2025-09-03                      283.251541  191.580069  251.181634   
2025-09-04                      283.251541  191.580069  251.181634   
2025-09-05                      283.251541  191.580069  251.181634   
2025-09-08                      283.251541  191.580069  251.181634   
2025-09-09                      283.251541  191.580069  251.181634   
2025-09-10                      283.251541  191.580069  251.181634   
2025-09-11                      283.251541  191.580069  251.181634   
2025-09-12                      283.251541  191.580069  251.181634   
2025-09-15                      283.251541  191.580069  251.181634   
2025-09-16                      283.251541  191.580069  251.181634   
2025-09-17                      283.251541  191.580069  251.181634   
2025-09-18                      283.251541  191.580069  251.181634   
2025-09-19                      283.251541  191.580069  251.181634   
2025-09-22                      283.251541  191.580069  251.181634   
2025-09-23                      283.251541  191.580069  251.181634   
2025-09-24                      283.251541  191.580069  251.181634   
2025-09-25                      283.251541  191.580069  251.181634   
2025-09-26                      283.251541  191.580069  251.181634   
2025-09-29                      283.251541  191.580069  251.181634   
2025-09-30                      283.251541  191.580069  251.181634   
2025-10-01                      283.251541  191.580069  251.181634   
2025-10-02                      283.251541  191.580069  251.181634   
2025-10-03                      283.251541  191.580069  251.181634   
2025-10-06                      283.251541  180.373872  251.181634   
2025-10-07                      283.251541  180.373872  251.181634   
2025-10-08                      283.251541  180.373872  251.181634   
2025-10-09                      283.251541  180.373872  217.984670   
2025-10-10                      283.251541  180.373872  191.740413   
2025-10-13                      283.251541  180.373872  191.740413   
2025-10-14                      283.251541  180.373872  191.740413   
2025-10-15                      283.251541  180.373872  191.740413   
2025-10-16                      283.251541  180.373872  191.740413   
2025-10-17                      283.251541  180.373872  191.740413   
2025-10-20                      275.667211  180.373872  191.740413   
2025-10-21                      275.667211  180.373872  191.740413   
2025-10-22

In [108]:
Cc.tail(50)

matrix_series_price_period              20                          \
matrix_series_supResPeriod              50                           
matrix_series_supResPercentage         100                           
matrix_series_smoother                   5                           
symbol                                 ANV         GVR         NKG   
date                                                                 
2025-10-21                       25.715196  -59.909452  -55.549283   
2025-10-22                        2.049602  -79.774153  -70.136970   
2025-10-23                      -15.879596  -90.244834  -83.529853   
2025-10-24                      -37.550642  -88.256860  -98.592711   
2025-10-27                      -59.840599  -44.719912 -117.496708   
2025-10-28                      -81.517163   -4.080491 -126.519836   
2025-10-29                      -94.354075   32.899229 -106.779376   
2025-10-30                      -79.501407   65.948266  -82.875607   
2025-10-31                      -48.790136  114.461596  -66.615862   
2025-11-03                      -34.693249  139.243895  -24.127159   
2025-11-04                      -51.715956  140.909138   28.771804   
2025-11-05                      -63.509592  158.018759   77.538724   
2025-11-06                      -72.661027  175.554508  104.610599   
2025-11-07                      -87.760735  137.403708  118.394950   
2025-11-10                     -102.599870  121.130280   94.700757   
2025-11-11                     -110.551405   95.751984   70.532602   
2025-11-12                     -105.272453   71.940511   75.264280   
2025-11-13                      -67.572969   61.738043   88.867163   
2025-11-14                      -36.020674   60.738190  101.171445   
2025-11-17                        5.751317   81.847524  114.068950   
2025-11-18                       59.981635   82.350439  144.497948   
2025-11-19                      102.279812   66.043364  155.527046   
2025-11-20                      116.253191   50.380512  123.041552   
2025-11-21                      116.674255   28.678872  101.954944   
2025-11-24                      116.223151    4.523052   71.922102   
2025-11-25                       87.159455  -26.523529   37.020857   
2025-11-26                       77.381966  -56.287240    6.358779   
2025-11-27                       60.872608  -72.569360  -20.331665   
2025-11-28                       38.918863  -84.953620  -49.125214   
2025-12-01                       19.021123 -101.991395  -80.365133   
2025-12-02                       -7.435748 -111.591385 -110.304506   
2025-12-03                      -37.397449  -99.867852 -131.964538   
2025-12-04                      -62.462825  -55.968012 -134.481445   
2025-12-05                      -84.349918  -36.460638 -109.480838   
2025-12-08                     -109.926880  -51.158846 -114.124886   
2025-12-09                     -134.316134  -67.771336 -125.471323   
2025-12-10                     -152.300269  -74.604605 -129.656105   
2025-12-11                     -165.507552  -85.059288 -132.940657   
2025-12-12                     -182.477925 -103.774688 -143.222826   
2025-12-15                     -199.461470 -123.931166 -156.596122   
2025-12-16                     -208.579755 -134.194854 -165.135729   
2025-12-17                     -198.994511 -129.121140 -163.801478   
2025-12-18                     -156.717653  -95.252968 -148.589288   
2025-12-19                     -124.156925  -61.789210 -126.904492   
2025-12-22                      -66.251117  -14.748567  -77.752754   
2025-12-23                      -24.897381   -4.847938  -29.581609   
2025-12-24                      -28.600618  -28.796892    1.673771   
2025-12-25                      -47.720028  -49.576716   -4.168303   
2025-12-26                      -61.365701  -64.850794   25.969238   
2025-12-29                      -70.186394  -76.305414   45.546280   

matrix_series_price_period                              
matrix_series_supResPe

In [ ]:
# pip install vectorbt yfinance pandas numpy

import numpy as np
import pandas as pd
import vectorbt as vbt
import yfinance as yf


# -----------------------------
# Helpers (AFL equivalents)
# -----------------------------
def ref(s: pd.Series, n: int) -> pd.Series:
    """AFL Ref(s, n): shift by n bars. In AFL, Ref(x, -1) means previous bar."""
    return s.shift(-n)

def wilders(s: pd.Series, n: int) -> pd.Series:
    """AFL Wilders smoothing ~ EMA with alpha=1/n, adjust=False."""
    return s.ewm(alpha=1 / n, adjust=False).mean()

def hhv(s: pd.Series, n: int) -> pd.Series:
    """AFL HHV: highest high value over last n bars."""
    return s.rolling(n, min_periods=n).max()

def llv(s: pd.Series, n: int) -> pd.Series:
    """AFL LLV: lowest low value over last n bars."""
    return s.rolling(n, min_periods=n).min()

def crossed_above(a: pd.Series, b) -> pd.Series:
    """AFL Cross(a,b): a crosses above b."""
    b = b if isinstance(b, pd.Series) else pd.Series(b, index=a.index)
    return (a > b) & (a.shift(1) <= b.shift(1))

def crossed_below(a: pd.Series, b) -> pd.Series:
    """Convenience: a crosses below b."""
    b = b if isinstance(b, pd.Series) else pd.Series(b, index=a.index)
    return (a < b) & (a.shift(1) >= b.shift(1))


# -----------------------------
# Parameters (from AFL defaults)
# -----------------------------
SupResPeriod = 50          # LookBack Period
SupResPercentage = 100     # Percentage
PricePeriod = 16           # Price Period

OverBought = 200
OverSold = -200
Smoother = 5


# -----------------------------
# Load data (replace as needed)
# -----------------------------
symbol = "AAPL"
df = yf.download(symbol, auto_adjust=False, progress=False)
df = df.rename(columns=str.title)  # Open High Low Close Volume

open_ = df["Open"]
high = df["High"]
low = df["Low"]
close = df["Close"]
vol = df["Volume"]


# -----------------------------
# "Swing Sup/Res" section
# -----------------------------
Lookback = SupResPeriod
PerCent = SupResPercentage
Pds = PricePeriod

# AFL: Var = MACD(); (use MACD line)
macd = vbt.MACD.run(close).macd

# AFL:
# Up = IIf( Var > Ref( Var, -1 ), abs( Var - Ref( Var, -1 ) ), 0 );
# Dn = IIf( Var < Ref( Var, -1 ), abs( Var - Ref( Var, -1 ) ), 0 );
macd_prev = macd.shift(1)
Up = np.where(macd > macd_prev, (macd - macd_prev).abs(), 0.0)
Dn = np.where(macd < macd_prev, (macd - macd_prev).abs(), 0.0)
Up = pd.Series(Up, index=close.index)
Dn = pd.Series(Dn, index=close.index)

# AFL: Ut = Wilders( Up, Pds ); Dt = Wilders( Dn, Pds ); RSIt = 100 * ( Ut / ( Ut + Dt ) );
Ut = wilders(Up, Pds)
Dt = wilders(Dn, Pds)
RSIt = 100 * (Ut / (Ut + Dt))

# AFL: Osc = CCI( pds );
Osc = vbt.CCI.run(high, low, close, window=Pds).cci

Value1 = Osc
Value2 = hhv(Value1, Lookback)
Value3 = llv(Value1, Lookback)
Value4 = Value2 - Value3
Value5 = Value4 * (PerCent / 100.0)

ResistanceLine = Value3 + Value5
SupportLine = Value2 - Value5


# -----------------------------
# "Entry/Exit Detail" section (signals)
# -----------------------------
n = Smoother
ys1 = (high + low + close * 2) / 4.0

rk3 = vbt.EMA.run(ys1, window=n).ema
rk4 = ys1.rolling(n, min_periods=n).std(ddof=0)  # StDev

rk5 = (ys1 - rk3) * 200.0 / rk4
rk6 = vbt.EMA.run(rk5, window=n).ema

# AFL: UP = EMA( rk6, n );
UP_line = vbt.EMA.run(rk6, window=n).ema

# AFL:
# Buy  = Cross( up, OverSold );
# Sell = Cross( OverBought, up );
#
# In AFL, Cross(constant, up) effectively triggers when up crosses BELOW the constant.
entries = crossed_above(UP_line, OverSold)
exits = crossed_below(UP_line, OverBought)


# -----------------------------
# Backtest with vectorbt
# -----------------------------
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    init_cash=100_000,
    fees=0.0,
    slippage=0.0
)

print(pf.stats())


# -----------------------------
# Optional: quick plots
# -----------------------------
# Price + entry/exit markers
pf.plot().show()

# Show oscillator + derived support/resistance bands
(vbt.Chart()
 .plot(Osc, name="CCI (Osc)")
 .plot(ResistanceLine, name="ResistanceLine")
 .plot(SupportLine, name="SupportLine")
 .show()
)

In [ ]:
## Load datasource from deltalake and save it to local h5 file
REMOTE_HOST = "http://minio.phuchuynh.xyz"
storage_options = {
    "AWS_ACCESS_KEY_ID": "CzOwnLkEDXQy951AOqes",
    "AWS_SECRET_ACCESS_KEY": "fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S",
    "AWS_ENDPOINT_URL": REMOTE_HOST,
    "AWS_ALLOW_HTTP": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "AWS_REGION": 'us-east-1',
    "aws_conditional_put": "etag",
}        

dt = DeltaTable("s3://delta-table-storage/stocks", storage_options=storage_options)
dt.optimize.z_order(["date"])


{'numFilesAdded': 0,
 'numFilesRemoved': 0,
 'filesAdded': '{"avg":0.0,"max":0,"min":0,"totalFiles":0,"totalSize":0}',
 'filesRemoved': '{"avg":0.0,"max":0,"min":0,"totalFiles":0,"totalSize":0}',
 'partitionsOptimized': 0,
 'numBatches': 0,
 'totalConsideredFiles': 410,
 'totalFilesSkipped': 410,
 'preserveInsertionOrder': True}